In [7]:
import torch
import gymnasium as gym
import time
import torch.nn as nn

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Define same Q-network used during training

In [9]:
HIDDEN_SIZE = 128

class QNetwork(nn.Module):
    def __init__(self, obs_size, n_actions, hidden=HIDDEN_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, x):
        return self.net(x)


### Create environment in render mode human

In [10]:
ENV_NAME = "CartPole-v1"
env = gym.make(ENV_NAME, render_mode="human")
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n

### Load the trained model

In [11]:
policy_net = QNetwork(obs_size, n_actions).to(device)
policy_net.load_state_dict(torch.load("dqn_cartpole_model.pth",  map_location=device))
policy_net.eval()

C:\Users\DELL\AppData\Local\Temp\ipykernel_13924\2177798405.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  policy_net.load_state_dict(torch.load("dqn_cartpole_model.pth

QNetwork(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)

### Running model and rendering CartPole

In [12]:
NUM_EPISODES = 10

for ep in range(NUM_EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        # Convert state to tensor
        state_v = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        # Choose greedy action
        with torch.no_grad():
            action = int(policy_net(state_v).argmax().item())

        # Next step in environment
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        state = next_state
        total_reward += reward

        # Slowing down rendering
        time.sleep(0.02)

    print(f"Episode {ep + 1} | Total Reward: {total_reward}")

env.close()


Episode 1 | Total Reward: 500.0
Episode 2 | Total Reward: 500.0
Episode 3 | Total Reward: 500.0
Episode 4 | Total Reward: 500.0
Episode 5 | Total Reward: 500.0
Episode 6 | Total Reward: 500.0
Episode 7 | Total Reward: 500.0
Episode 8 | Total Reward: 500.0
Episode 9 | Total Reward: 500.0
Episode 10 | Total Reward: 500.0
